# Homework 2 — Part 1: Data cleaning and validation

This notebook inspects the intentionally messy `ohlc_hw` table, records a quality report **before** any repair, then writes a retained `ohlc_hw_cleaned` table in a **local working copy** of the database.

The course source file `Lectures/Lecture 4/data/data.db` is never modified. If `data/data.db` is missing, it is copied from that course path once.

In [1]:
from __future__ import annotations

import re
import shutil
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

cwd = Path.cwd().resolve()
if (cwd / "01_data_cleaning.ipynb").is_file():
    HW_DIR = cwd
else:
    hits = [
        path.parent for path in cwd.rglob("01_data_cleaning.ipynb")
        if (path.parent / "03_run_all_stocks.py").is_file()
    ]
    if not hits:
        raise FileNotFoundError(
            "Run this notebook from Homeworks/HW2/Suge_Li "
            "(the folder that contains 01_data_cleaning.ipynb)."
        )
    HW_DIR = hits[0]

COURSE_DB = next(
    (p / "Lectures/Lecture 4/data/data.db" for p in [HW_DIR, *HW_DIR.parents]
     if (p / "Lectures/Lecture 4/data/data.db").is_file()),
    None,
)
DB_PATH = HW_DIR / "data" / "data.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DB_PATH.is_file():
    if COURSE_DB is None:
        raise FileNotFoundError(
            "Place a local copy of data.db at data/data.db "
            "(download from bCourses; do not commit it)."
        )
    shutil.copy2(COURSE_DB, DB_PATH)
    print(f"Copied course database to working copy: {DB_PATH}")
else:
    print(f"Using working copy: {DB_PATH}")
print(f"Homework directory: {HW_DIR}")


Using working copy: /Users/lisuge/Developer/UCB-pre-MFE/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Suge_Li/data/data.db
Homework directory: /Users/lisuge/Developer/UCB-pre-MFE/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Suge_Li


## 1.1 Load and inspect

Columns in `ohlc_hw`: `ts` (observation date), `ticker`, raw `open`/`high`/`low`/`close`, vendor `adj_close`, and `volume`. The lecture examples sometimes use `date`; this homework keeps `ts` and converts it to ISO `YYYY-MM-DD` when writing the cleaned table.


In [2]:
conn = sqlite3.connect(DB_PATH)
raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
print("Loaded ohlc_hw from", DB_PATH)
raw.info()
display(raw.describe(include="all").T)
print("Date range (as stored):", raw["ts"].min(), "→", raw["ts"].max())
print("Raw ticker labels:", raw["ticker"].nunique())
display(pd.Series(sorted(raw["ticker"].astype(str).unique()), name="raw_ticker"))
print("Missing values in each column:\n", raw.isna().sum())


Loaded ohlc_hw from /Users/lisuge/Developer/UCB-pre-MFE/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Suge_Li/data/data.db
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ts,397711,15868,2020-11-24,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
open,397711.0,NaN,NaN,NaN,46.285479,67.132594,0.0,4.209938,19.85,57.53,604.26
high,397711.0,NaN,NaN,NaN,46.74766,67.77821,0.005208,4.24,20.06,58.13,608.63
low,397711.0,NaN,NaN,NaN,45.816279,66.477178,0.004801,4.16,19.63,56.95,597.88
close,397711.0,NaN,NaN,NaN,46.292903,67.141879,0.005208,4.2,19.86,57.53,605.4
adj_close,377856.0,NaN,NaN,NaN,51.65225,194.584045,0.002945,1.39,11.96,43.77,10442.510783
volume,397711.0,NaN,NaN,NaN,30977733.689946,126623650.175253,0.0,1730700.0,4648200.0,11023796.5,9276606399.0
ticker,397711,150,PG,15450,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Date range (as stored): 1962-01-02 → 2024-12-06
Raw ticker labels: 150


0      <a target="_blank"><span style="color: blue">A...
1      <a target="_blank"><span style="color: blue">A...
2      <a target="_blank"><span style="color: blue">A...
3      <a target="_blank"><span style="color: blue">A...
4      <a target="_blank"><span style="color: blue">B...
                             ...                        
145                                                  TRV
146                                                  UNH
147                                                    V
148                                                   VZ
149                                                  WMT
Name: raw_ticker, Length: 150, dtype: object

Missing values in each column:
 ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64


## 1.2 Detect data-quality issues

Ticker markup, whitespace, and case are normalized **before** grouping or deduplicating. Exact duplicate rows are removed. Conflicting `(ticker, ts)` keys are investigated rather than keeping the first row arbitrarily.


In [3]:
def normalize_ticker(series: pd.Series) -> pd.Series:
    """Strip HTML/markup, whitespace, and case from ticker labels."""
    text = series.astype("string").str.strip()
    extracted = text.str.extract(r">([A-Za-z0-9.\-]{1,10})</", expand=False)
    cleaned = extracted.fillna(text)
    cleaned = cleaned.str.replace(r"<[^>]+>", "", regex=True)
    cleaned = cleaned.str.replace(r"\s+", "", regex=True).str.upper()
    return cleaned.astype("string")


def sort_ticker_date(frame: pd.DataFrame, date_col: str = "ts") -> pd.DataFrame:
    """Sort by ticker then date before returns, fills, or gap checks."""
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    return out.sort_values(["ticker", date_col]).reset_index(drop=True)


def missing_sessions(frame: pd.DataFrame, expected_dates, date_col: str = "ts") -> pd.DatetimeIndex:
    """Return expected dates absent from one ticker's observations."""
    observed = pd.DatetimeIndex(pd.to_datetime(frame[date_col]).dropna().unique())
    return pd.DatetimeIndex(expected_dates).difference(observed).sort_values()


def simple_close_returns(frame: pd.DataFrame) -> pd.Series:
    """Simple raw-close returns for the Part 1 diagnostic screen."""
    ordered = sort_ticker_date(frame)
    valid = ordered["close"].where(np.isfinite(ordered["close"]) & ordered["close"].gt(0))
    returns = valid.groupby(ordered["ticker"], sort=False).pct_change(fill_method=None)
    returns.index = ordered.index
    return returns


def detect_issues(frame: pd.DataFrame, date_col: str = "ts") -> pd.DataFrame:
    """Build a quality report. Multiple issues per observation are allowed.

    Severity choices:
    - high: missing/invalid prices, OHLC bound failures, conflicting keys
    - medium: missing sessions, missing adj_close, zero volume, adj mismatch
    - low: extreme-return review flags (not a deletion rule)
    """
    data = sort_ticker_date(frame, date_col=date_col)
    price_cols = ["open", "high", "low", "close"]
    raw_prices = data[price_cols]
    adj = data["adj_close"]
    returns = simple_close_returns(data)
    grouped = returns.groupby(data["ticker"], sort=False)
    mean = grouped.transform("mean")
    std = grouped.transform(lambda x: x.std(ddof=1))
    finite_std = std.gt(0) & np.isfinite(std)

    ohlc_bounds = (
        data["high"].lt(data[["open", "close"]].max(axis=1))
        | data["low"].gt(data[["open", "close"]].min(axis=1))
        | data["high"].lt(data["low"])
    )
    masks = {
        "missing_raw_price": raw_prices.isna().any(axis=1),
        "invalid_raw_price": (~np.isfinite(raw_prices) | raw_prices.le(0)).any(axis=1),
        "missing_adj_close": adj.isna(),
        "invalid_adj_close": adj.notna() & (~np.isfinite(adj) | adj.le(0)),
        "ohlc_bounds": ohlc_bounds,
        "high_below_close": data["high"].lt(data["close"]),
        "low_above_close": data["low"].gt(data["close"]),
        "invalid_volume": ~np.isfinite(data["volume"]) | data["volume"].lt(0),
        "zero_volume": data["volume"].eq(0),
        "extreme_return": returns.notna() & finite_std & (returns - mean).abs().gt(3 * std),
        "zero_or_undefined_return_std": returns.notna() & ~finite_std,
    }
    descriptions = {
        "missing_raw_price": "At least one raw OHLC field is missing",
        "invalid_raw_price": "Nonpositive or non-finite raw OHLC price",
        "missing_adj_close": "Adjusted close is missing",
        "invalid_adj_close": "Adjusted close is nonpositive or non-finite",
        "ohlc_bounds": "Fails low <= open,close <= high",
        "high_below_close": "Impossible high < close",
        "low_above_close": "Impossible low > close",
        "invalid_volume": "Volume is negative or non-finite",
        "zero_volume": "Zero volume (possible stale quote; not treated as missing)",
        "extreme_return": "Simple close return more than 3 sample SDs from the ticker mean (review flag)",
        "zero_or_undefined_return_std": "Return SD is zero or undefined; no statistical outlier flags applied",
    }
    severity = {
        "missing_raw_price": "high",
        "invalid_raw_price": "high",
        "missing_adj_close": "medium",
        "invalid_adj_close": "high",
        "ohlc_bounds": "high",
        "high_below_close": "high",
        "low_above_close": "high",
        "invalid_volume": "high",
        "zero_volume": "medium",
        "extreme_return": "low",
        "zero_or_undefined_return_std": "low",
    }
    parts = []
    for issue, mask in masks.items():
        rows = data.loc[mask, ["ticker", date_col]].rename(columns={date_col: "issue_date"})
        if rows.empty:
            continue
        rows["issue_type"] = issue
        rows["description"] = descriptions[issue]
        rows["severity"] = severity[issue]
        parts.append(rows)
    if not parts:
        return pd.DataFrame(columns=["ticker", "issue_type", "issue_date", "description", "severity"])
    report = pd.concat(parts, ignore_index=True)
    report["issue_date"] = pd.to_datetime(report["issue_date"]).dt.strftime("%Y-%m-%d")
    return report[["ticker", "issue_type", "issue_date", "description", "severity"]]


def missing_session_issues(
    frame: pd.DataFrame,
    reference_calendar: pd.DataFrame,
    date_col: str = "ts",
) -> pd.DataFrame:
    """Gaps versus the supplied ohlc reference calendar for each ticker."""
    data = sort_ticker_date(frame, date_col=date_col)
    ref = reference_calendar.copy()
    ref["ts"] = pd.to_datetime(ref["ts"])
    ref["ticker"] = normalize_ticker(ref["ticker"])
    rows = []
    for ticker, group in data.groupby("ticker", sort=True):
        ticker_ref = ref.loc[ref["ticker"] == ticker, "ts"]
        if ticker_ref.empty:
            print(f"Ticker {ticker}: no reference calendar; gap check skipped")
            continue
        start, end = ticker_ref.min(), ticker_ref.max()
        expected = pd.DatetimeIndex(
            ticker_ref[(ticker_ref >= start) & (ticker_ref <= end)].unique()
        ).sort_values()
        gaps = missing_sessions(group, expected, date_col=date_col)
        if len(gaps) == 0:
            print(f"Ticker {ticker} is missing 0 trading days between {start.date()} and {end.date()}")
            continue
        print(f"Ticker {ticker} is missing {len(gaps)} trading days between {start.date()} and {end.date()}")
        for gap in gaps:
            rows.append({
                "ticker": ticker,
                "issue_type": "missing_session",
                "issue_date": gap.strftime("%Y-%m-%d"),
                "description": f"Absent from supplied ohlc reference calendar ({start.date()} to {end.date()})",
                "severity": "medium",
            })
    if not rows:
        return pd.DataFrame(columns=["ticker", "issue_type", "issue_date", "description", "severity"])
    return pd.DataFrame(rows)


### Tiny injected examples

The supplied homework table can correctly show **zero** missing sessions. The detector is demonstrated on a one-day gap, and repair behavior is shown on synthetic invalid rows that are not present in `ohlc_hw`.


In [4]:
expected_sessions = pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"])
calendar_delivery = pd.DataFrame({
    "ticker": ["DEMO_A"] * 3,
    "ts": expected_sessions[[0, 2, 3]],
    "open": [10.0, 10.2, 10.1],
    "high": [10.5, 10.6, 10.4],
    "low": [9.8, 10.0, 9.9],
    "close": [10.1, 10.3, 10.2],
    "adj_close": [10.1, 10.3, np.nan],
    "volume": [1000, np.nan, 1100],
})
gap_dates = missing_sessions(calendar_delivery, expected_sessions)
print(
    f"Ticker DEMO_A is missing {len(gap_dates)} trading days between "
    f"{expected_sessions.min().date()} and {expected_sessions.max().date()}: {list(gap_dates.date)}"
)
assert gap_dates.tolist() == [pd.Timestamp("2024-01-03")]

injected = pd.DataFrame({
    "ticker": ["DEMO_B"] * 5,
    "ts": pd.to_datetime(["2024-02-01", "2024-02-02", "2024-02-03", "2024-02-04", "2024-02-05"]),
    "open": [100.0, 101.0, np.nan, 0.0, 104.0],
    "high": [102.0, 103.0, np.nan, 106.0, 107.0],
    "low": [99.0, 100.0, np.nan, 103.0, 103.5],
    "close": [101.0, 102.0, np.nan, 105.0, 80.0],  # last close is inside range but a large drop
    "adj_close": [101.0, 102.0, np.nan, 105.0, 80.0],
    "volume": [1_000_000, -5, np.nan, 0, 900_000],
})
injected.loc[4, "close"] = 200.0  # high < close
injected_report = detect_issues(injected)
display(injected_report)
assert set(injected_report["issue_type"]).issuperset({
    "missing_raw_price", "invalid_raw_price", "ohlc_bounds", "high_below_close",
    "invalid_volume", "zero_volume",
})
print("Injected-example detector checks passed.")


Ticker DEMO_A is missing 1 trading days between 2024-01-02 and 2024-01-05: [datetime.date(2024, 1, 3)]


,ticker,issue_type,issue_date,description,severity
0,DEMO_B,missing_raw_price,2024-02-03,At least one raw OHLC field is missing,high
1,DEMO_B,invalid_raw_price,2024-02-03,Nonpositive or non-finite raw OHLC price,high
2,DEMO_B,invalid_raw_price,2024-02-04,Nonpositive or non-finite raw OHLC price,high
3,DEMO_B,missing_adj_close,2024-02-03,Adjusted close is missing,medium
4,DEMO_B,ohlc_bounds,2024-02-04,"Fails low <= open,close <= high",high
5,DEMO_B,ohlc_bounds,2024-02-05,"Fails low <= open,close <= high",high
6,DEMO_B,high_below_close,2024-02-05,Impossible high < close,high
7,DEMO_B,invalid_volume,2024-02-02,Volume is negative or non-finite,high
8,DEMO_B,invalid_volume,2024-02-03,Volume is negative or non-finite,high
9,DEMO_B,zero_volume,2024-02-04,Zero volume (possible stale quote; not treated...,medium


Injected-example detector checks passed.


### Normalize, exact-dedup, then diagnose the real table


In [5]:
work = raw.copy()
work["ticker_raw"] = work["ticker"].astype("string")
work["ticker"] = normalize_ticker(work["ticker"])
print("Normalized tickers:", sorted(work["ticker"].dropna().unique()))
print("Exact duplicate rows:", int(work.duplicated().sum()))

exact_dup_mask = work.duplicated(keep="first")
removed_exact_dups = int(exact_dup_mask.sum())
work = work.loc[~exact_dup_mask].copy()
work = sort_ticker_date(work)

key_counts = work.groupby(["ticker", "ts"]).size()
conflicting = key_counts[key_counts > 1]
print("Remaining (ticker, ts) keys with more than one row:", int(len(conflicting)))
if len(conflicting):
    display(work.set_index(["ticker", "ts"]).loc[conflicting.index].head(20))
    raise ValueError("Conflicting keys remain; they need an explicit source-selection rule.")

reference = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn)
reference["ts"] = pd.to_datetime(reference["ts"])
reference["ticker"] = normalize_ticker(reference["ticker"])

pre_repair_report = pd.concat(
    [detect_issues(work), missing_session_issues(work, reference)],
    ignore_index=True,
)
# Date-specific adj factor from ohlc, used only as an audit of vendor adj_close.
factor_lookup = reference.rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
merged_audit = work.merge(factor_lookup, on=["ticker", "ts"], how="left")
trusted_adj = merged_audit["close"] * (merged_audit["ref_adj"] / merged_audit["ref_close"])
adj_mismatch = (
    merged_audit["adj_close"].notna()
    & trusted_adj.notna()
    & ~np.isclose(merged_audit["adj_close"], trusted_adj, rtol=1e-4, atol=1e-6)
)
mismatch_rows = merged_audit.loc[adj_mismatch, ["ticker", "ts"]].rename(columns={"ts": "issue_date"})
if not mismatch_rows.empty:
    mismatch_rows = mismatch_rows.assign(
        issue_type="adj_close_inconsistent",
        description="Vendor adj_close disagrees with the matched ohlc date-specific factor * raw close",
        severity="medium",
    )
    mismatch_rows["issue_date"] = pd.to_datetime(mismatch_rows["issue_date"]).dt.strftime("%Y-%m-%d")
    pre_repair_report = pd.concat(
        [pre_repair_report, mismatch_rows[["ticker", "issue_type", "issue_date", "description", "severity"]]],
        ignore_index=True,
    )

print("Quality report rows (pre-repair):", len(pre_repair_report))
display(pre_repair_report.groupby(["issue_type", "severity"]).size().rename("n"))
display(pre_repair_report.head(12))

Normalized tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']
Exact duplicate rows: 29460
Remaining (ticker, ts) keys with more than one row: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 19

/var/folders/g5/c11574kj41xfz2m8y1ps8lgw0000gn/T/ipykernel_40060/2250446951.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pre_repair_report = pd.concat(


issue_type              severity
adj_close_inconsistent  medium      10417
extreme_return          low          5056
invalid_raw_price       high            5
missing_adj_close       medium      18374
ohlc_bounds             high            5
zero_volume             medium         75
Name: n, dtype: int64

,ticker,issue_type,issue_date,description,severity
0,CVX,invalid_raw_price,1962-01-02,Nonpositive or non-finite raw OHLC price,high
1,JNJ,invalid_raw_price,1962-01-02,Nonpositive or non-finite raw OHLC price,high
2,MCD,invalid_raw_price,1966-07-05,Nonpositive or non-finite raw OHLC price,high
3,MRK,invalid_raw_price,1962-01-02,Nonpositive or non-finite raw OHLC price,high
4,PG,invalid_raw_price,1962-01-02,Nonpositive or non-finite raw OHLC price,high
5,AAPL,missing_adj_close,1981-01-08,Adjusted close is missing,medium
6,AAPL,missing_adj_close,1981-01-13,Adjusted close is missing,medium
7,AAPL,missing_adj_close,1981-02-12,Adjusted close is missing,medium
8,AAPL,missing_adj_close,1981-04-16,Adjusted close is missing,medium
9,AAPL,missing_adj_close,1981-05-19,Adjusted close is missing,medium


## 1.3 Clean the data

Design choices:

1. **Missing raw prices** — drop a row only if *all* of open, high, low, and close are missing *before* filling. Remaining missing prices are forward-filled **within ticker** after sorting. Leading gaps are quarantined rather than back-filled or borrowed from another ticker.
2. **Missing volume** — fill with 0 and set `volume_unknown`. Zero is not evidence of no trading.
3. **Impossible OHLC** — Option 3: invalidate the offending field(s), then reconstruct from the other prices on the same day when they form a valid range. If today's range cannot host yesterday's close, yesterday is **not** copied (forward-fill alone can violate today's high/low).
4. **Extreme returns (> 3 SD)** — review flags only. In this database the raw close matches the supplied `ohlc` table, so these moves are retained rather than interpolated. Interpolation is reserved for confirmed field errors.
5. **Adjusted close** — reconstruct missing or inconsistent `adj_close` as `validated_close * (ohlc.adj_close / ohlc.close)` on the same `(ticker, ts)`. That is a date-specific factor, not a factor carried across an unverified corporate action. If close was repaired and the factor cannot be verified, the row is quarantined. Raw OHLC is never replaced by the reference table.


In [6]:
RAW_PRICE_COLS = ["open", "high", "low", "close"]


def _repair_ohlc_row(open_, high, low, close):
    """Reconstruct a single day's raw OHLC from remaining valid fields."""
    vals = {"open": open_, "high": high, "low": low, "close": close}
    finite_positive = {
        name: value for name, value in vals.items()
        if np.isfinite(value) and value > 0
    }
    if not finite_positive:
        return vals, False

    # If high/low are usable, clip open/close into that range.
    high_ok = "high" in finite_positive
    low_ok = "low" in finite_positive
    if high_ok and low_ok and finite_positive["high"] < finite_positive["low"]:
        # Swap an inverted range when both exist but are reversed.
        finite_positive["high"], finite_positive["low"] = finite_positive["low"], finite_positive["high"]
        high_ok = low_ok = True

    if high_ok and low_ok:
        lo, hi = finite_positive["low"], finite_positive["high"]
        for name in ("open", "close"):
            if name in finite_positive:
                finite_positive[name] = float(np.clip(finite_positive[name], lo, hi))
            else:
                # Prefer the other bound-respecting price; else the midpoint.
                other = finite_positive.get("close" if name == "open" else "open")
                finite_positive[name] = float(other if other is not None else (lo + hi) / 2)
        vals.update({k: finite_positive[k] for k in RAW_PRICE_COLS})
        ok = lo <= vals["open"] <= hi and lo <= vals["close"] <= hi and hi >= lo
        return vals, bool(ok)

    # Not enough range information: use the previous valid close later via ffill.
    return {name: finite_positive.get(name, np.nan) for name in RAW_PRICE_COLS}, False


def clean_ohlc(frame: pd.DataFrame, reference_adj: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Return (cleaned, quarantined, counts). Raw table is not modified."""
    data = sort_ticker_date(frame)
    counts = {
        "rows_in": int(len(data)),
        "dropped_all_raw_missing": 0,
        "quarantined": 0,
        "retained": 0,
    }
    data["ticker_normalized"] = data["ticker_raw"].astype(str).ne(data["ticker"].astype(str))
    data["volume_unknown"] = ~np.isfinite(data["volume"]) | data["volume"].isna()
    for col in RAW_PRICE_COLS:
        data[f"{col}_repaired"] = False
    data["adj_reconstructed"] = False
    data["extreme_return_review"] = False
    data["zero_volume"] = False
    data["quarantine_reason"] = pd.Series(pd.NA, index=data.index, dtype="string")

    all_raw_missing = data[RAW_PRICE_COLS].isna().all(axis=1)
    counts["dropped_all_raw_missing"] = int(all_raw_missing.sum())
    dropped = data.loc[all_raw_missing].assign(quarantine_reason="all_raw_prices_missing")
    data = data.loc[~all_raw_missing].copy().reset_index(drop=True)

    for col in RAW_PRICE_COLS:
        bad = ~np.isfinite(data[col]) | data[col].le(0)
        data.loc[bad, f"{col}_repaired"] = True
        data.loc[bad, col] = np.nan

    before = data[RAW_PRICE_COLS].copy()
    repaired = [_repair_ohlc_row(o, h, lo, c)[0] for o, h, lo, c in
                zip(data["open"], data["high"], data["low"], data["close"])]
    data["open"] = [row["open"] for row in repaired]
    data["high"] = [row["high"] for row in repaired]
    data["low"] = [row["low"] for row in repaired]
    data["close"] = [row["close"] for row in repaired]
    for col in RAW_PRICE_COLS:
        changed = ~np.isclose(
            pd.to_numeric(data[col], errors="coerce"),
            pd.to_numeric(before[col], errors="coerce"),
            rtol=0.0, atol=0.0, equal_nan=True,
        )
        data.loc[changed, f"{col}_repaired"] = True

    for col in RAW_PRICE_COLS:
        was_na = data[col].isna()
        data[col] = data.groupby("ticker", sort=False)[col].ffill()
        data.loc[was_na & data[col].notna(), f"{col}_repaired"] = True

    invalid_volume = data["volume"].lt(0) | ~np.isfinite(data["volume"]) | data["volume"].isna()
    data.loc[invalid_volume, "volume_unknown"] = True
    data.loc[data["volume_unknown"], "volume"] = 0
    data["zero_volume"] = data["volume"].eq(0)

    bounds_bad = (
        data["high"].lt(data[["open", "close"]].max(axis=1))
        | data["low"].gt(data[["open", "close"]].min(axis=1))
        | data["high"].lt(data["low"])
    )
    for idx in data.index[bounds_bad]:
        vals, ok = _repair_ohlc_row(
            data.at[idx, "open"], data.at[idx, "high"],
            data.at[idx, "low"], data.at[idx, "close"],
        )
        for col in RAW_PRICE_COLS:
            current = data.at[idx, col]
            if pd.isna(current) or pd.isna(vals[col]) or not np.isclose(current, vals[col]):
                data.at[idx, f"{col}_repaired"] = True
            data.at[idx, col] = vals[col]
        if not ok:
            data.at[idx, "quarantine_reason"] = "unresolved_ohlc_bounds"

    rets = simple_close_returns(data[["ticker", "ts", "close"]])
    mean = rets.groupby(data["ticker"], sort=False).transform("mean")
    std = rets.groupby(data["ticker"], sort=False).transform(lambda x: x.std(ddof=1))
    data["extreme_return_review"] = (
        rets.notna() & std.gt(0) & np.isfinite(std) & (rets - mean).abs().gt(3 * std)
    ).fillna(False).astype(bool)

    ref = reference_adj.rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
    data = data.merge(ref, on=["ticker", "ts"], how="left")
    factor = data["ref_adj"] / data["ref_close"]
    trusted = data["close"] * factor
    needs_adj = (
        data["adj_close"].isna()
        | ~np.isfinite(data["adj_close"])
        | data["adj_close"].le(0)
        | (
            trusted.notna()
            & data["adj_close"].notna()
            & ~np.isclose(data["adj_close"].to_numpy(dtype=float), trusted.to_numpy(dtype=float),
                          rtol=1e-4, atol=1e-6)
        )
    )
    can_rebuild = needs_adj & trusted.notna() & np.isfinite(trusted) & trusted.gt(0)
    data.loc[can_rebuild, "adj_close"] = trusted[can_rebuild]
    data.loc[can_rebuild, "adj_reconstructed"] = True
    matches_factor = (
        trusted.notna() & np.isfinite(trusted) & trusted.gt(0)
        & np.isclose(
            data["adj_close"].to_numpy(dtype=float),
            trusted.to_numpy(dtype=float),
            rtol=1e-4, atol=1e-6, equal_nan=False,
        )
    )
    unresolved_adj = (needs_adj & ~can_rebuild) | (data["close_repaired"] & ~matches_factor)
    data.loc[unresolved_adj, "quarantine_reason"] = data.loc[unresolved_adj, "quarantine_reason"].fillna(
        "unresolved_adj_close"
    )

    leading = data[RAW_PRICE_COLS + ["adj_close"]].isna().any(axis=1)
    data.loc[leading, "quarantine_reason"] = data.loc[leading, "quarantine_reason"].fillna(
        "leading_or_unfilled_price"
    )
    still_bad_prices = (
        ~np.isfinite(data[RAW_PRICE_COLS + ["adj_close"]]).all(axis=1)
        | data[RAW_PRICE_COLS + ["adj_close"]].le(0).any(axis=1)
    )
    data.loc[still_bad_prices, "quarantine_reason"] = data.loc[still_bad_prices, "quarantine_reason"].fillna(
        "nonpositive_or_nonfinite_after_repair"
    )
    data = data.drop(columns=["ref_close", "ref_adj"])

    keep_mask = data["quarantine_reason"].isna()
    cleaned = data.loc[keep_mask].copy()
    quarantined = data.loc[~keep_mask].copy()
    if not dropped.empty:
        quarantined = pd.concat([dropped, quarantined], ignore_index=True, sort=False)

    cleaned["ts"] = pd.to_datetime(cleaned["ts"]).dt.strftime("%Y-%m-%d")
    if not quarantined.empty:
        quarantined["ts"] = pd.to_datetime(quarantined["ts"]).dt.strftime("%Y-%m-%d")

    counts["quarantined"] = int(len(quarantined))
    counts["retained"] = int(len(cleaned))
    return cleaned.reset_index(drop=True), quarantined.reset_index(drop=True), counts


# Demonstrate repair on the injected example (no ohlc match → adj quarantine if unrestorable).
demo_ref = pd.DataFrame({
    "ticker": ["DEMO_B", "DEMO_B"],
    "ts": pd.to_datetime(["2024-02-01", "2024-02-02"]),
    "close": [101.0, 102.0],
    "adj_close": [101.0, 102.0],
})
demo_in = injected.copy()
demo_in["ticker_raw"] = demo_in["ticker"]
demo_clean, demo_q, demo_counts = clean_ohlc(demo_in, demo_ref)
print("Injected repair counts:", demo_counts)
display(demo_clean)
display(demo_q[["ticker", "ts", "quarantine_reason"]] if len(demo_q) else demo_q)


Injected repair counts: {'rows_in': 5, 'dropped_all_raw_missing': 1, 'quarantined': 2, 'retained': 3}


,ticker,ts,open,high,low,close,adj_close,volume,ticker_raw,ticker_normalized,volume_unknown,open_repaired,high_repaired,low_repaired,close_repaired,adj_reconstructed,extreme_return_review,zero_volume,quarantine_reason
0,DEMO_B,2024-02-01,100.0,102.0,99.0,101.0,101.0,1000000.0,DEMO_B,False,False,False,False,False,False,False,False,False,<NA>
1,DEMO_B,2024-02-02,101.0,103.0,100.0,102.0,102.0,0.0,DEMO_B,False,True,False,False,False,False,False,False,True,<NA>
2,DEMO_B,2024-02-04,105.0,106.0,103.0,105.0,105.0,0.0,DEMO_B,False,False,True,False,False,False,False,False,True,<NA>


,ticker,ts,quarantine_reason
0,DEMO_B,2024-02-03,all_raw_prices_missing
1,DEMO_B,2024-02-05,unresolved_adj_close


In [7]:
cleaned, quarantined, counts = clean_ohlc(work, reference)
print("Cleaning counts:", counts)
print("Removed exact duplicates before clean():", removed_exact_dups)
display(cleaned.head())
print("Repair flag totals:")
flag_cols = [
    "ticker_normalized", "volume_unknown", "open_repaired", "high_repaired",
    "low_repaired", "close_repaired", "adj_reconstructed", "extreme_return_review", "zero_volume",
]
display(cleaned[flag_cols].sum())
print("Quarantined rows:", len(quarantined))
if len(quarantined):
    display(quarantined["quarantine_reason"].value_counts())


Cleaning counts: {'rows_in': 368251, 'dropped_all_raw_missing': 0, 'quarantined': 0, 'retained': 368251}
Removed exact duplicates before clean(): 29460


,ts,open,high,low,close,adj_close,volume,ticker,ticker_raw,ticker_normalized,volume_unknown,open_repaired,high_repaired,low_repaired,close_repaired,adj_reconstructed,extreme_return_review,zero_volume,quarantine_reason
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,AAPL,False,False,False,False,False,False,False,False,False,<NA>
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,AAPL,False,False,False,False,False,False,False,False,False,<NA>
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,AAPL,False,False,False,False,False,False,False,False,False,<NA>
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,AAPL,False,False,False,False,False,False,False,False,False,<NA>
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,AAPL,False,False,False,False,False,False,False,False,False,<NA>


Repair flag totals:


ticker_normalized        36985
volume_unknown               0
open_repaired                5
high_repaired                0
low_repaired                 0
close_repaired               0
adj_reconstructed        28791
extreme_return_review     5056
zero_volume                 75
dtype: int64

Quarantined rows: 0


### Validation, write `ohlc_hw_cleaned`, and before/after stats

`ts` is stored as ISO `YYYY-MM-DD`. The original `ohlc_hw` table is left unchanged.


In [8]:
def validate_cleaned(frame: pd.DataFrame, reference_adj: pd.DataFrame) -> None:
    prices = frame[["open", "high", "low", "close", "adj_close"]]
    assert np.isfinite(prices.to_numpy()).all(), "non-finite prices remain"
    assert (prices > 0).all().all(), "nonpositive prices remain"
    assert not prices.isna().any().any(), "missing prices remain"
    assert np.isfinite(frame["volume"]).all() and (frame["volume"] >= 0).all()
    bounds_ok = (
        frame["low"].le(frame["open"]) & frame["open"].le(frame["high"])
        & frame["low"].le(frame["close"]) & frame["close"].le(frame["high"])
        & frame["low"].le(frame["high"])
    )
    assert bounds_ok.all(), "OHLC bounds still fail"
    assert not frame.duplicated(["ticker", "ts"]).any(), "duplicate keys remain"
    ts = pd.to_datetime(frame["ts"], format="%Y-%m-%d", errors="raise")
    assert ts.notna().all()

    # Recheck adjusted-price consistency after repair.
    ref = reference_adj.rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
    check = frame.assign(ts=pd.to_datetime(frame["ts"])).merge(ref, on=["ticker", "ts"], how="left")
    trusted = check["close"] * (check["ref_adj"] / check["ref_close"])
    mismatch = (
        check["adj_close"].notna() & trusted.notna()
        & ~np.isclose(check["adj_close"].to_numpy(dtype=float),
                      trusted.to_numpy(dtype=float), rtol=1e-4, atol=1e-6)
    )
    assert not mismatch.any(), f"{int(mismatch.sum())} adj_close rows still inconsistent with reference"


validate_cleaned(cleaned, reference)
post_report = detect_issues(cleaned.assign(ts=pd.to_datetime(cleaned["ts"])))
post_report = pd.concat(
    [post_report, missing_session_issues(cleaned.assign(ts=pd.to_datetime(cleaned["ts"])), reference)],
    ignore_index=True,
)
# Remaining expected issues: extreme-return review flags and zero volume (retained by design).
print("Post-repair issue types:")
display(post_report.groupby("issue_type").size().rename("n") if len(post_report) else post_report)
unexpected = post_report.loc[~post_report["issue_type"].isin(["extreme_return", "zero_volume", "missing_session"])]
assert unexpected.empty, unexpected.head()
print("Post-repair validation passed (only review flags / zero volume remain).")

print("\nBefore (raw ohlc_hw) vs after (retained cleaned rows):")
before_after = pd.DataFrame({
    "raw_rows": [len(raw)],
    "exact_dups_removed": [removed_exact_dups],
    "quarantined": [counts["quarantined"]],
    "retained": [counts["retained"]],
    "raw_tickers_labels": [raw["ticker"].nunique()],
    "cleaned_tickers": [cleaned["ticker"].nunique()],
    "raw_adj_missing": [int(raw["adj_close"].isna().sum())],
    "cleaned_adj_missing": [int(cleaned["adj_close"].isna().sum())],
    "raw_min_open": [float(pd.to_numeric(raw["open"]).min())],
    "cleaned_min_open": [float(cleaned["open"].min())],
}).T.rename(columns={0: "value"})
display(before_after)

write_cols = [
    "ts", "ticker", "open", "high", "low", "close", "adj_close", "volume",
    "ticker_raw", *flag_cols,
]
cleaned_out = cleaned[write_cols].copy()
cleaned_out.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
if len(quarantined):
    quarantined.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)
else:
    pd.DataFrame(columns=write_cols + ["quarantine_reason"]).to_sql(
        "ohlc_hw_quarantine", conn, if_exists="replace", index=False
    )

# Keep raw table unchanged: compare row count.
raw_count = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw", conn)["n"].iloc[0]
cleaned_count = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw_cleaned", conn)["n"].iloc[0]
print(f"ohlc_hw rows unchanged: {raw_count} (original load {len(raw)})")
print(f"ohlc_hw_cleaned rows: {cleaned_count}")
assert raw_count == len(raw)
conn.close()
print("Wrote ohlc_hw_cleaned and ohlc_hw_quarantine.")


Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

/var/folders/g5/c11574kj41xfz2m8y1ps8lgw0000gn/T/ipykernel_40060/241652439.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  post_report = pd.concat(


issue_type
extreme_return    5056
zero_volume         75
Name: n, dtype: int64

Post-repair validation passed (only review flags / zero volume remain).

Before (raw ohlc_hw) vs after (retained cleaned rows):


,value
raw_rows,397711.000000
exact_dups_removed,29460.000000
quarantined,0.000000
retained,368251.000000
raw_tickers_labels,150.000000
cleaned_tickers,30.000000
raw_adj_missing,19855.000000
cleaned_adj_missing,0.000000
raw_min_open,0.000000
cleaned_min_open,0.005208


ohlc_hw rows unchanged: 397711 (original load 397711)
ohlc_hw_cleaned rows: 368251
Wrote ohlc_hw_cleaned and ohlc_hw_quarantine.
